# FORCE 2020 well-log QC and lithology workflow

This concise notebook calls the tested `welllog` package. It uses complete wells for model evaluation and defaults to a small subset for an interactive demonstration. Run the CLI command in the README for the verified 118-well result.

In [ ]:
from pathlib import Path

from welllog.io import load_las_directory
from welllog.pipeline import run_pipeline, write_results
from welllog.plotting import plot_confusion, plot_log_tracks

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
LAS_DIRECTORY = PROJECT_ROOT / 'data' / 'raw' / 'Force_2020_all_wells_train_test_blind_hidden_final'
ARTIFACT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'notebook-demo'
MAX_WELLS = 8
RANDOM_STATE = 42

## Load a bounded set of wells

The loader normalizes curve names, converts null sentinels, preserves the well identifier, and sorts each well by measured depth.

In [ ]:
all_rows = load_las_directory(LAS_DIRECTORY)
selected_names = sorted(all_rows['WELL'].unique())[:MAX_WELLS]
demo = all_rows.loc[all_rows['WELL'].isin(selected_names)].reset_index(drop=True)
{'wells': demo['WELL'].nunique(), 'rows': len(demo), 'selected': selected_names}

## Train with wells held out

All gradients and rolling statistics reset at well boundaries. Human interpretation confidence is retained for analysis but excluded from training features.

In [ ]:
result = run_pipeline(demo, test_size=0.25, random_state=RANDOM_STATE, max_iter=40)
write_results(result, ARTIFACT_DIRECTORY)
result.metrics

In [ ]:
plot_confusion(result.confusion, normalize=True);

## Inspect one held-out well

The saved predictions include measured inference curves so the same artifact can drive a dashboard without reopening the raw LAS files or retraining.

In [ ]:
review_well = result.test_wells[0]
review = result.predictions.query('WELL == @review_well')
curves = [name for name in ('GR', 'RHOB', 'NPHI', 'DTC') if name in review and review[name].notna().any()]
plot_log_tracks(review, curves, title=f'Held-out well {review_well}');